# Cubic invariant irreps: Cartesian tensor decomposition vs direct Reynolds projection

This notebook compares two ways of creating crystal-symmetry-invariant irrep features for the cubic groups:

- **`O`**: the proper octahedral/cubic rotation group, 24 rotations, determinant `+1`.
- **`Oh`**: the full octahedral/cubic group, 48 orthogonal operations, obtained as `O ∪ (-O)`.

We focus first on the FCC/cubic case used by the OCRP encoder.

The two methods are:

1. **Cartesian tensor → trace/decomposition route**  
   Build a rank-4 Cartesian tensor from a cubic orbit, then decompose it into SO(3) irreps such as `0e`, `2e`, and `4e`.

2. **Direct Reynolds route**  
   Work directly inside a chosen SO(3) irrep `le`, average the representation matrices over the crystal group, and extract the invariant eigenspace.

The central thing to keep separate:

- A **crystal-invariant seed** is fixed by the right action of the crystal group.
- The final orientation feature is usually **SO(3)-equivariant**, not globally invariant: rotating the sample rotates the feature in its irrep space.

## 0. Setup

This notebook uses the same repo implementation that builds the OCRP local-isometric embedding:

- `models/local_iso_embedding.py`
- especially `_build_nonscalar_projector`, `cubic_group_O`, and the rank-4 FCC block.

Run from the repository root, or let the cell below add the root to `sys.path`.

In [1]:
from pathlib import Path
import json
import math
import sys

repo = Path.cwd()
for cand in [repo, *repo.parents]:
    if (cand / "models" / "local_iso_embedding.py").exists():
        repo = cand
        break
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
from e3nn import o3

from models.local_iso_embedding import (
    cubic_group_O,
    dihedral_group_D6_zaxis,
    full_symmetry_formula,
    _build_nonscalar_projector,
    _quat_conjugate,
    _quat_to_matrix_active,
)

torch.set_printoptions(precision=5, sci_mode=True, linewidth=120)
dtype = torch.float64
device = torch.device("cpu")

print("repo:", repo)

repo: /data/home/umang/Materials/Reynolds-QSR_paper


## 1. The objective: right-invariant, left-equivariant features

Let `R` be the active orientation matrix used internally by the encoder.

For cubic crystal symmetry, `R` and `R g` describe the same physical crystal orientation whenever `g ∈ O` is a cubic symmetry operation. Therefore the feature should satisfy

$$
f(Rg)=f(R), \qquad g\in O.
$$

But if the whole sample is physically rotated by `h ∈ SO(3)`, the feature should not stay numerically fixed; it should transform equivariantly:

$$
f(hR)=D^{(l)}(h)f(R).
$$

So in this notebook, "invariant irrep" means:

> an SO(3) irrep-valued feature whose **seed** is invariant under the crystal subgroup.

For FCC/O, the production rank-4 embedding ultimately produces a `4e` irrep vector of dimension `2*4+1 = 9`.

## 2. Build `O` and `Oh`

The proper cubic group `O` has 24 rotations. The full cubic group `Oh` includes inversion, so we can represent it as

$$
O_h = O \cup (-O).
$$

For an even-parity irrep like `4e`, inversion acts as `+1`, so `O` and `Oh` give the same invariant seed. For odd-parity irreps, inversion acts as `-1`, so averaging over `Oh` cancels the seed.

In [2]:
G_O = cubic_group_O(dtype=dtype, device=device)  # [24, 3, 3]
G_Oh = torch.cat([G_O, -G_O], dim=0)             # [48, 3, 3]

print("|O|  =", G_O.shape[0])
print("|Oh| =", G_Oh.shape[0])
print("det(O) unique:", torch.unique(torch.round(torch.linalg.det(G_O))))
print("det(Oh) unique:", torch.unique(torch.round(torch.linalg.det(G_Oh))))

|O|  = 24
|Oh| = 48
det(O) unique: tensor([1.00000e+00], dtype=torch.float64)
det(Oh) unique: tensor([-1.00000e+00, 1.00000e+00], dtype=torch.float64)


# Method A: Cartesian tensor product, trace, and decomposition

This is the method used by the current local-isometric OCRP encoder.

For FCC, the code chooses:

$$
u=e_1=(1,0,0), \qquad \text{rank}=4.
$$

At orientation `R`, it constructs the group-averaged tensor

$$
T_4(R)=\frac{1}{|O|}\sum_{g\in O}(Rgu)^{\otimes 4}.
$$

At the reference orientation `R=I`, this becomes

$$
T_4(I)=\frac{1}{24}\sum_{g\in O}(ge_1)^{\otimes 4}.
$$

In code, the orbit directions are built in `LocalIsoCrystalEmbedding.__init__` by

```python
orbit_directions = self.group_mats @ spec.reference_direction
```

and the runtime tensor-power average is

```python
_tensor_power_flat_fast(v, rank).mean(dim=-2)
```

## 3. FCC orbit of `e1`

Every proper cubic rotation sends `e1` to one of the six coordinate-axis directions:

$$
\pm e_1,\quad \pm e_2,\quad \pm e_3.
$$

Each one occurs four times among the 24 rotations.

In [3]:
e1 = torch.tensor([1.0, 0.0, 0.0], dtype=dtype, device=device)
orbit = G_O @ e1

unique, counts = torch.unique(orbit.round(decimals=6), dim=0, return_counts=True)
print("unique orbit directions and multiplicities")
for v, c in zip(unique, counts):
    print(f"{v.tolist()}  count={int(c)}")

unique orbit directions and multiplicities
[-1.0, 0.0, 0.0]  count=4
[0.0, -1.0, 0.0]  count=4
[0.0, 0.0, -1.0]  count=4
[0.0, 0.0, 1.0]  count=4
[-0.0, 1.0, 0.0]  count=4
[1.0, 0.0, 0.0]  count=4


Because the rank is four, signs disappear:

$$
(-e_i)^{\otimes4}=e_i^{\otimes4}.
$$

Therefore

$$
T_4(I)=\frac13\left(e_1^{\otimes4}+e_2^{\otimes4}+e_3^{\otimes4}\right).
$$

This is already much less general than an arbitrary symmetric rank-4 tensor.

In [4]:
# Explicitly build T_4(I) = mean_g (g e1)^{⊗4}
T4 = torch.einsum("ga,gb,gc,gd->abcd", orbit, orbit, orbit, orbit) / orbit.shape[0]

nonzero = []
for i in range(3):
    for j in range(3):
        for k in range(3):
            for l in range(3):
                val = T4[i, j, k, l].item()
                if abs(val) > 1e-12:
                    nonzero.append(((i+1, j+1, k+1, l+1), val))

print("nonzero components of T4(I):")
for idx, val in nonzero:
    print(f"T{idx} = {val:.8f}")

nonzero components of T4(I):
T(1, 1, 1, 1) = 0.33333333
T(2, 2, 2, 2) = 0.33333333
T(3, 3, 3, 3) = 0.33333333


## 4. Why the `2e` part is zero: one trace

A symmetric rank-4 tensor decomposes as

$$
\mathrm{Sym}^4(\mathbb R^3)=0e\oplus2e\oplus4e.
$$

The `2e` content can be detected by taking one trace:

$$
A_{ij}=\sum_{k=1}^3 T_{ijkk}.
$$

This gives a symmetric 3-by-3 matrix. Its traceless part is the `2e` piece:

$$
A^{(2)}=A-\frac{\mathrm{tr}(A)}{3}I.
$$

For the cubic tensor, this should be zero because the three axes occur equally.

In [5]:
# A_ij = sum_k T_ijkk
A = T4.diagonal(dim1=2, dim2=3).sum(dim=-1)
A_iso = torch.trace(A) / 3.0 * torch.eye(3, dtype=dtype, device=device)
A_2e = A - A_iso

print("A = trace over last two indices of T4:")
print(A)
print("\ntr(A)/3 * I:")
print(A_iso)
print("\nA^(2) = A - tr(A)/3 I:")
print(A_2e)
print("\n||A^(2)|| =", A_2e.norm().item())

A = trace over last two indices of T4:
tensor([[3.33333e-01, 9.12304e-49, 9.12304e-49],
        [9.12304e-49, 3.33333e-01, 0.00000e+00],
        [9.12304e-49, 0.00000e+00, 3.33333e-01]], dtype=torch.float64)

tr(A)/3 * I:
tensor([[3.33333e-01, 0.00000e+00, 0.00000e+00],
        [0.00000e+00, 3.33333e-01, 0.00000e+00],
        [0.00000e+00, 0.00000e+00, 3.33333e-01]], dtype=torch.float64)

A^(2) = A - tr(A)/3 I:
tensor([[0.00000e+00, 9.12304e-49, 9.12304e-49],
        [9.12304e-49, 0.00000e+00, 0.00000e+00],
        [9.12304e-49, 0.00000e+00, 0.00000e+00]], dtype=torch.float64)

||A^(2)|| = 1.8246073754229388e-48


Interpretation:

- The trace matrix is proportional to the identity: `A = I/3`.
- Its traceless part is zero.
- Therefore there is no `2e` information.

The tensor is not fully isotropic, though. Its fourth-order anisotropy remains, and that is the nonzero `4e` component.

## 5. e3nn CartesianTensor decomposition used by the code

The production code does not manually compute `A_ij = T_ijkk`.

Instead, it builds a generic linear projector

$$
\text{flattened Cartesian rank-4 tensor}\rightarrow 2e\oplus4e
$$

using `e3nn.io.CartesianTensor`.

The scalar `0e` is intentionally dropped because it is constant for unit directions and carries no orientation information.

In [6]:
proj, irreps_no_scalar = _build_nonscalar_projector(
    rank=4,
    formula=full_symmetry_formula(4),
    dtype=dtype,
    device=device,
)

y = T4.reshape(1, -1) @ proj

print("irreps after dropping scalar:", irreps_no_scalar)
print("feature shape:", tuple(y.shape))
print()
for sl, mul_ir in zip(irreps_no_scalar.slices(), irreps_no_scalar):
    block = y[0, sl]
    print(f"{mul_ir}: norm={block.norm().item():.8e}, max_abs={block.abs().max().item():.8e}")

irreps after dropping scalar: 1x2e+1x4e
feature shape: (1, 14)

1x2e: norm=4.88199639e-18, max_abs=4.86223382e-18
1x4e: norm=3.65148369e-01, max_abs=2.78886676e-01


The output should show:

- `1x2e` is numerical zero.
- `1x4e` is nonzero.

This is why the full generic rank-4 projector reports `2e + 4e`, but the actual cubic orbit only has useful `4e` content.

# Method B: direct Reynolds projection in irrep space

The direct Reynolds method skips the Cartesian tensor entirely.

Pick an SO(3) irrep degree `l`, for example `l=4`. Let `D^l(g)` be the irrep matrix for group element `g`. Then define

$$
P_l=\frac1{|G|}\sum_{g\in G}D^l(g).
$$

This is the Reynolds projector. Its eigenvectors with eigenvalue `1` span the subspace fixed by every group element:

$$
D^l(g)u=u,\qquad g\in G.
$$

Those `u` vectors are the crystal-invariant seeds inside the chosen irrep.

In [7]:
def reynolds_projector(group_mats, l: int, parity: str = "e"):
    """Return P_l = mean_g D^l_parity(g)."""
    ir = o3.Irrep(f"{l}{parity}")
    Dg = ir.D_from_matrix(group_mats)
    P = Dg.mean(dim=0)
    # Numerical symmetrization; exact P is an orthogonal projector.
    return 0.5 * (P + P.T)


def invariant_basis(group_mats, l: int, parity: str = "e", tol: float = 1e-7):
    P = reynolds_projector(group_mats, l=l, parity=parity)
    evals, evecs = torch.linalg.eigh(P)
    mask = evals > (1.0 - tol)
    U = evecs[:, mask]
    return P, evals, U


for l in range(0, 9):
    _, evals, U = invariant_basis(G_O, l=l, parity="e")
    print(f"O, {l}e: invariant_dim={U.shape[1]}, top_eval={evals[-1].item(): .8f}")

O, 0e: invariant_dim=1, top_eval= 1.00000000
O, 1e: invariant_dim=0, top_eval= 0.00000008
O, 2e: invariant_dim=0, top_eval= 0.00000002
O, 3e: invariant_dim=0, top_eval= 0.00000012
O, 4e: invariant_dim=1, top_eval= 1.00000000
O, 5e: invariant_dim=0, top_eval= 0.00000017
O, 6e: invariant_dim=1, top_eval= 1.00000000
O, 7e: invariant_dim=0, top_eval= 0.00000017
O, 8e: invariant_dim=1, top_eval= 1.00000000


For `O`, the direct Reynolds method sees invariant seeds at

$$
l=0,4,6,8,\ldots
$$

up to the range checked above.

This is a key difference from the rank-4 Cartesian method:

- Direct Reynolds asks: **does this chosen `l` contain a cubic-invariant seed?**
- The rank-4 Cartesian tensor asks: **which `l` values arise from this particular rank-4 orbit tensor?**

A rank-4 symmetric tensor can only produce `0e`, `2e`, and `4e`; it cannot produce `6e` even though direct Reynolds says cubic `6e` invariant seeds exist.

## 6. `O` versus `Oh`

`Oh` includes inversion. In e3nn notation:

- `le` has even parity, so inversion acts as `+1`.
- `lo` has odd parity, so inversion acts as `-1`.

The FCC rank-4 tensor has even parity, so it lives in `0e`, `2e`, `4e`. For these even-parity irreps, averaging over `Oh` gives the same answer as averaging over `O`.

Odd parity irreps are killed by the inversion half of `Oh`.

In [8]:
print("Even parity over Oh")
for l in range(0, 9):
    _, evals, U = invariant_basis(G_Oh, l=l, parity="e")
    print(f"Oh, {l}e: invariant_dim={U.shape[1]}, top_eval={evals[-1].item(): .8f}")

print("\nOdd parity over Oh")
for l in range(0, 9):
    _, evals, U = invariant_basis(G_Oh, l=l, parity="o")
    print(f"Oh, {l}o: invariant_dim={U.shape[1]}, top_eval={evals[-1].item(): .8e}")

Even parity over Oh
Oh, 0e: invariant_dim=1, top_eval= 1.00000000
Oh, 1e: invariant_dim=0, top_eval= 0.00000008
Oh, 2e: invariant_dim=0, top_eval= 0.00000002
Oh, 3e: invariant_dim=0, top_eval= 0.00000012
Oh, 4e: invariant_dim=1, top_eval= 1.00000000
Oh, 5e: invariant_dim=0, top_eval= 0.00000017
Oh, 6e: invariant_dim=1, top_eval= 1.00000000
Oh, 7e: invariant_dim=0, top_eval= 0.00000017
Oh, 8e: invariant_dim=1, top_eval= 1.00000000

Odd parity over Oh
Oh, 0o: invariant_dim=0, top_eval= 0.00000000e+00
Oh, 1o: invariant_dim=0, top_eval= 9.14824383e-18
Oh, 2o: invariant_dim=0, top_eval= 1.63168294e-17
Oh, 3o: invariant_dim=0, top_eval= 1.92345486e-17
Oh, 4o: invariant_dim=0, top_eval= 3.93754518e-17
Oh, 5o: invariant_dim=0, top_eval= 2.00832585e-17
Oh, 6o: invariant_dim=0, top_eval= 1.92330898e-17
Oh, 7o: invariant_dim=0, top_eval= 2.36644800e-17
Oh, 8o: invariant_dim=0, top_eval= 4.60984599e-17


So, for the current FCC rank-4 OCRP embedding:

$$
O \text{ and } O_h \text{ are equivalent because the descriptor has even parity.}
$$

If we were working with odd-parity features, `Oh` would matter strongly.

## 7. Do both methods produce the same cubic `4e` seed?

Yes. For the FCC rank-4 block:

1. The Cartesian method builds `T4(I)` and projects it to `4e`.
2. The direct Reynolds method finds the one-dimensional invariant eigenspace inside `4e`.

Those two vectors should be the same up to sign and normalization.

In [9]:
# Cartesian seed: take the 4e block from the decomposed T4(I)
slices = list(irreps_no_scalar.slices())
labels = [str(mul_ir) for mul_ir in irreps_no_scalar]
idx_4e = labels.index("1x4e")
cart_seed_4e = y[0, slices[idx_4e]]
cart_seed_4e = cart_seed_4e / cart_seed_4e.norm()

# Reynolds seed: eigenvector of P_4 with eigenvalue 1
_, evals4, U4 = invariant_basis(G_O, l=4, parity="e")
reyn_seed_4e = U4[:, 0]
reyn_seed_4e = reyn_seed_4e / reyn_seed_4e.norm()

cosine = torch.abs(torch.dot(cart_seed_4e, reyn_seed_4e)).item()

print("cartesian 4e seed norm:", cart_seed_4e.norm().item())
print("reynolds  4e seed norm:", reyn_seed_4e.norm().item())
print("abs cosine similarity:", cosine)
print("top eigenvalue of P_4:", evals4[-1].item())

cartesian 4e seed norm: 1.0
reynolds  4e seed norm: 1.0
abs cosine similarity: 0.9999999999999981
top eigenvalue of P_4: 0.9999999999998324


The absolute cosine should be essentially `1.0`. The sign is arbitrary because an eigenvector can be multiplied by `-1` without changing the invariant line.

This tells us:

$$
\text{Cartesian rank-4 FCC orbit} \quad \Longrightarrow \quad \text{same } 4e \text{ A1 line as direct Reynolds.}
$$

## 8. Real data check: dataset quaternions start passive

The derivations above used the reference tensor `T4(I)` and group representation matrices. Now we check the two methods on actual FCC data.

For the real dataset used here, the stored quaternions are treated as:

- scalar-first: `[w, x, y, z]`;
- passive/Bunge: sample/specimen frame `→` crystal frame.

The encoder therefore converts each real quaternion to the active convention by conjugation:

$$
q_\mathrm{active}=\overline{q_\mathrm{passive}}=[w,-x,-y,-z].
$$

Then it builds the active matrix `R(q_active)` and evaluates both constructions on the same real orientations.

In [10]:
def to_hwc4(arr: np.ndarray) -> np.ndarray:
    """Return a quaternion image as (H, W, 4), accepting (H,W,4) or (4,H,W)."""
    arr = np.asarray(arr)
    if arr.ndim != 3:
        raise ValueError(f"expected rank-3 quaternion image, got shape={arr.shape}")
    if arr.shape[-1] == 4:
        return arr
    if arr.shape[0] == 4:
        return np.moveaxis(arr, 0, -1)
    raise ValueError(f"could not find quaternion axis in shape={arr.shape}")


def find_real_fcc_passive_file():
    """Find a real FCC/IN718 quaternion block from the local dataset mount."""
    candidate_roots = [
        Path("/data/home/umang/Materials/Materials_data_mount/datasets/IN718_QSR_x4"),
        Path("/data/home/umang/Materials/Materials_data_mount/datasets/h200_datasets/IN718_QSR_x4"),
    ]
    for root in candidate_roots:
        info_path = root / "dataset_info.json"
        if not info_path.exists():
            continue
        for split in ("Test", "Val", "Train"):
            for which in ("HR_Data", "LR_Data"):
                files = sorted((root / split / which).glob("*.npy"))
                if files:
                    return root, info_path, files[0]
    raise FileNotFoundError(
        "Could not find IN718_QSR_x4 under the expected dataset roots. "
        "Update candidate_roots in this cell to point at your FCC dataset."
    )


real_root, real_info_path, real_npy_path = find_real_fcc_passive_file()
real_info = json.loads(real_info_path.read_text())
real_arr = np.load(real_npy_path)
real_q_hwc = to_hwc4(real_arr).astype(np.float64, copy=False)

print("dataset root:", real_root)
print("dataset_info:", real_info_path)
print("sample file:", real_npy_path)
print("symmetry:", real_info.get("symmetry"))
print("formatting:", real_info.get("formatting"))
print("raw array shape:", real_arr.shape, "-> HWC shape:", real_q_hwc.shape)

# The dataset metadata records scalar-first storage. The passive convention is the
# convention assumed by the repo's encoder and baseline adapters.
q_real_passive_all = torch.from_numpy(real_q_hwc.reshape(-1, 4)).to(dtype=dtype, device=device)
q_real_passive_all = q_real_passive_all / q_real_passive_all.norm(dim=-1, keepdim=True).clamp_min(1e-12)
q_real_passive_all = torch.where(q_real_passive_all[..., :1] < 0.0, -q_real_passive_all, q_real_passive_all)

# Use a deterministic subset so the notebook is quick but genuinely data-driven.
gen = torch.Generator(device="cpu")
gen.manual_seed(123)
n_real = min(512, q_real_passive_all.shape[0])
real_idx = torch.randperm(q_real_passive_all.shape[0], generator=gen)[:n_real]
q_real_passive = q_real_passive_all[real_idx]

# Passive/Bunge -> active for the irrep machinery.
q_real_active = _quat_conjugate(q_real_passive)
R_real = _quat_to_matrix_active(q_real_active)

print("sampled real passive quaternions:", tuple(q_real_passive.shape))
print("active rotation matrices:", tuple(R_real.shape))
print("first passive q:", q_real_passive[0])
print("first active  q:", q_real_active[0])

dataset root: /data/home/umang/Materials/Materials_data_mount/datasets/IN718_QSR_x4
dataset_info: /data/home/umang/Materials/Materials_data_mount/datasets/IN718_QSR_x4/dataset_info.json
sample file: /data/home/umang/Materials/Materials_data_mount/datasets/IN718_QSR_x4/Test/HR_Data/IN718_QSR_x4_test_hr_x_block_104.npy
symmetry: Oh
formatting: {'Note:': 'The original quaternions stored in the Original_Data files were converted to same scalar-first or last convention, depending on the convert_to_scalar_first flag.', 'convert_to_scalar_first': True, 'normalize': True, 'hemisphere': True, 'reduce_fz': True, 'to_quat_first': False}
raw array shape: (256, 256, 4) -> HWC shape: (256, 256, 4)
sampled real passive quaternions: (512, 4)
active rotation matrices: (512, 3, 3)
first passive q: tensor([9.36648e-01, -5.47845e-02, -1.46528e-01, 3.13398e-01], dtype=torch.float64)
first active  q: tensor([9.36648e-01, 5.47845e-02, 1.46528e-01, -3.13398e-01], dtype=torch.float64)


The next cell evaluates:

1. **Cartesian route on real data**

$$
T_4(R)=\frac1{24}\sum_{g\in O}(Rge_1)^{\otimes4}\quad\rightarrow\quad4e.
$$

2. **Direct Reynolds route on real data**

$$
f_4(R)=D^{(4)}(R)u_4,
$$

where `u4` is the invariant seed from the Reynolds projector.

The two outputs should match up to floating-point error after using the same seed normalization and sign.

In [11]:
def tensor_power_flat_rank4(v: torch.Tensor) -> torch.Tensor:
    """Same rank-4 flattening idea as _tensor_power_flat_fast, specialized for this check."""
    p2 = (v.unsqueeze(-1) * v.unsqueeze(-2)).reshape(*v.shape[:-1], 9)
    p4 = (p2.unsqueeze(-1) * p2.unsqueeze(-2)).reshape(*v.shape[:-1], 81)
    return p4


# --- Cartesian route: real passive q -> active R -> orbit tensor -> CartesianTensor projector.
orbit_t = orbit.transpose(0, 1).contiguous()  # [3, 24]
v_real = torch.matmul(R_real, orbit_t).transpose(-2, -1)  # [N, 24, 3]
x_real = tensor_power_flat_rank4(v_real).mean(dim=-2)      # [N, 81]
y_real = x_real @ proj                                    # [N, 14] = 2e + 4e

cart_seed_raw_4e = y[0, slices[idx_4e]]
cart_real_4e = y_real[:, slices[idx_4e]] / cart_seed_raw_4e.norm().clamp_min(1e-12)

# --- Direct Reynolds route: real passive q -> active R -> D^4(R) u4.
ir4 = o3.Irrep("4e")
reyn_seed_4e_aligned = reyn_seed_4e.clone()
if torch.dot(cart_seed_4e, reyn_seed_4e_aligned) < 0.0:
    reyn_seed_4e_aligned = -reyn_seed_4e_aligned
reyn_real_4e = ir4.D_from_matrix(R_real) @ reyn_seed_4e_aligned

diff = cart_real_4e - reyn_real_4e
print("real-data Cartesian vs Reynolds 4e")
print("  max abs error:", diff.abs().max().item())
print("  RMS error    :", diff.square().mean().sqrt().item())

# Also show that the nominal 2e block is zero on real orientations.
idx_2e = labels.index("1x2e")
real_2e = y_real[:, slices[idx_2e]]
print("\nreal-data Cartesian 2e residual")
print("  max abs:", real_2e.abs().max().item())
print("  RMS    :", real_2e.square().mean().sqrt().item())

print("\nfirst real-data 4e feature, Cartesian:")
print(cart_real_4e[0])
print("first real-data 4e feature, Reynolds:")
print(reyn_real_4e[0])

real-data Cartesian vs Reynolds 4e
  max abs error: 9.29943468730432e-07
  RMS error    : 3.065333297077292e-07

real-data Cartesian 2e residual
  max abs: 2.442817482516579e-09
  RMS    : 8.035324740679547e-10

first real-data 4e feature, Cartesian:
tensor([3.93169e-01, -1.56878e-01, -2.17991e-01, 3.92710e-01, -1.67294e-01, 1.35207e-01, -6.09443e-01, -3.96131e-01, 2.10849e-01],
       dtype=torch.float64)
first real-data 4e feature, Reynolds:
tensor([3.93169e-01, -1.56877e-01, -2.17991e-01, 3.92710e-01, -1.67294e-01, 1.35207e-01, -6.09444e-01, -3.96131e-01, 2.10849e-01],
       dtype=torch.float64)


## 8b. Numerical error budget for the FCC comparison

In exact real arithmetic, the two FCC constructions are the same object written two ways.

The Cartesian route computes

$$
\Pi_{4e}\left[\frac{1}{|O|}\sum_{g\in O}(Rge_1)^{\otimes4}\right],
$$

where $\Pi_{4e}$ is the `CartesianTensor` projection onto the `4e` block.

The direct Reynolds route computes

$$
D^{(4)}(R)u_4,
$$

where $u_4$ is the invariant vector satisfying

$$
D^{(4)}(g)u_4=u_4\qquad \forall g\in O.
$$

Because the seed is invariant, these are exactly equal after choosing the same normalization and sign:

$$
\Pi_{4e}\left[\frac{1}{|O|}\sum_{g\in O}(Rge_1)^{\otimes4}\right]
=
D^{(4)}(R)u_4.
$$

So any nonzero difference printed by the notebook is numerical error, not a mathematical discrepancy. The main finite-precision sources are:

1. real dataset quaternions are stored numerically, then normalized and conjugated;
2. `R(q)` is built in floating point, so it is only approximately orthogonal;
3. `CartesianTensor` and `Irrep.D_from_matrix` use numerical bases for the same irrep space;
4. the Reynolds projector is built by averaging floating-point Wigner-D matrices and extracting an eigenvector;
5. tensor powers and group averages involve many multiply-add operations.

Important convention note: this error check does **not** prove the passive convention. If we skip the passive-to-active conjugation, the Cartesian and Reynolds routes will still agree with each other because both are fed the same wrong matrix. The convention is justified by the repo pathway and dataset interpretation; the numerical comparison only verifies that the two invariant-irrep construction methods agree once the same active matrix is chosen.


In [12]:
# Numerical diagnostics explaining the FCC residuals above.
# These values should be tiny. They locate the residual in floating-point construction,
# not in a failure of the invariant-irrep identity.

I3 = torch.eye(3, dtype=dtype, device=device)
I9 = torch.eye(9, dtype=dtype, device=device)

R_real_orth_err = (R_real @ R_real.transpose(-1, -2) - I3).abs().max().item()
R_real_det_err = (torch.linalg.det(R_real) - 1.0).abs().max().item()

P4_O = reynolds_projector(G_O, l=4, parity="e")
P4_idempotence_err = (P4_O @ P4_O - P4_O).abs().max().item()

cart_seed_residual = (P4_O @ cart_seed_4e - cart_seed_4e).norm().item()
reyn_seed_residual = (P4_O @ reyn_seed_4e_aligned - reyn_seed_4e_aligned).norm().item()
seed_l2_mismatch = (cart_seed_4e - reyn_seed_4e_aligned).norm().item()
seed_max_mismatch = (cart_seed_4e - reyn_seed_4e_aligned).abs().max().item()

D4_real = ir4.D_from_matrix(R_real)
D4_orth_err = (D4_real.transpose(-1, -2) @ D4_real - I9).abs().max().item()

feature_diff = cart_real_4e - reyn_real_4e

# If the stored passive quaternion were mistakenly read as active, both routes would still
# agree with each other, but the produced physical feature would change.
R_real_if_passive_misread_as_active = _quat_to_matrix_active(q_real_passive)
v_wrong = torch.matmul(R_real_if_passive_misread_as_active, orbit_t).transpose(-2, -1)
x_wrong = tensor_power_flat_rank4(v_wrong).mean(dim=-2)
y_wrong = x_wrong @ proj
cart_wrong_4e = y_wrong[:, slices[idx_4e]] / cart_seed_raw_4e.norm().clamp_min(1e-12)
reyn_wrong_4e = ir4.D_from_matrix(R_real_if_passive_misread_as_active) @ reyn_seed_4e_aligned
wrong_internal_diff = cart_wrong_4e - reyn_wrong_4e
wrong_vs_correct_shift = cart_wrong_4e - cart_real_4e

print("FCC numerical error budget")
print(f"  dtype machine epsilon                         : {torch.finfo(dtype).eps:.3e}")
print(f"  max ||R R^T - I||_∞ after passive->active      : {R_real_orth_err:.3e}")
print(f"  max |det(R)-1| after passive->active           : {R_real_det_err:.3e}")
print(f"  max ||D4(R)^T D4(R)-I||_∞                      : {D4_orth_err:.3e}")
print(f"  max ||P4^2-P4||_∞                              : {P4_idempotence_err:.3e}")
print(f"  ||P4 cart_seed - cart_seed||_2                 : {cart_seed_residual:.3e}")
print(f"  ||P4 reyn_seed - reyn_seed||_2                 : {reyn_seed_residual:.3e}")
print(f"  ||cart_seed - aligned_reyn_seed||_2            : {seed_l2_mismatch:.3e}")
print(f"  max |cart_seed - aligned_reyn_seed|            : {seed_max_mismatch:.3e}")
print(f"  observed feature max_abs error                 : {feature_diff.abs().max().item():.3e}")
print(f"  observed feature RMS error                     : {feature_diff.square().mean().sqrt().item():.3e}")
print(f"  observed FCC 2e max_abs residual               : {real_2e.abs().max().item():.3e}")
print(f"  observed FCC 2e RMS residual                   : {real_2e.square().mean().sqrt().item():.3e}")

print("\nConvention guard")
print(f"  if passive q is misread as active, Cartesian-vs-Reynolds max_abs : {wrong_internal_diff.abs().max().item():.3e}")
print(f"  but wrong-convention feature shift vs correct active feature     : {wrong_vs_correct_shift.abs().max().item():.3e}")


FCC numerical error budget
  dtype machine epsilon                         : 2.220e-16
  max ||R R^T - I||_∞ after passive->active      : 4.441e-16
  max |det(R)-1| after passive->active           : 4.441e-16
  max ||D4(R)^T D4(R)-I||_∞                      : 2.753e-14
  max ||P4^2-P4||_∞                              : 7.201e-08
  ||P4 cart_seed - cart_seed||_2                 : 6.308e-08
  ||P4 reyn_seed - reyn_seed||_2                 : 1.679e-13
  ||cart_seed - aligned_reyn_seed||_2            : 6.308e-08
  max |cart_seed - aligned_reyn_seed|            : 6.044e-08
  observed feature max_abs error                 : 9.299e-07
  observed feature RMS error                     : 3.065e-07
  observed FCC 2e max_abs residual               : 2.443e-09
  observed FCC 2e RMS residual                   : 8.035e-10

Convention guard
  if passive q is misread as active, Cartesian-vs-Reynolds max_abs : 1.597e-06
  but wrong-convention feature shift vs correct active feature     : 1.259e+00


## 9. From invariant seed to orientation feature

Once we have a crystal-invariant seed `u_l`, the orientation feature is

$$
f_l(R)=D^l(R)u_l.
$$

Right invariance follows because

$$
f_l(Rg)=D^l(Rg)u_l=D^l(R)D^l(g)u_l=D^l(R)u_l=f_l(R).
$$

Left equivariance follows because

$$
f_l(hR)=D^l(hR)u_l=D^l(h)f_l(R).
$$

The current OCRP encoder reaches this behavior through Cartesian orbit tensors; the older direct method reaches it by explicitly constructing `u_l` via Reynolds projection.

In [13]:
def random_unit_quats(n, seed=0, dtype=torch.float64):
    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)
    q = torch.randn(n, 4, generator=gen, dtype=dtype)
    q = q / q.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    return torch.where(q[..., :1] < 0, -q, q)

# Check f_4(Rg) = f_4(R) using the Reynolds seed.
q = random_unit_quats(64, seed=11, dtype=dtype)
R = o3.quaternion_to_matrix(q)

# Pick one nontrivial cubic symmetry g.
g = G_O[5]

ir4 = o3.Irrep("4e")
f_R = ir4.D_from_matrix(R) @ reyn_seed_4e
f_Rg = ir4.D_from_matrix(R @ g) @ reyn_seed_4e

print("max |f(Rg) - f(R)|:", (f_Rg - f_R).abs().max().item())

max |f(Rg) - f(R)|: 5.638370780269852e-07


## 9b. Local-isometry preparation for `O`

The local-isometry step is a **metric preparation step on the final feature map**

$$
\Phi(R) = D^{(l)}(R) u_l
$$

or on a concatenation of such blocks. It does **not** depend on whether the seed `u_l` was found by Cartesian tensor decomposition or by direct Reynolds projection.

What matters is the final map `Phi`:

- If the Cartesian tensor route and the Reynolds route produce the same invariant seed line/subspace, they have the same local metric up to a fixed basis change and scaling.
- If they choose different invariant seed subspaces or different block weights, they define different embeddings and must be calibrated separately.

For the FCC `O` rank-4 case here, both routes produce the same `4e` A1 line. The cell below computes the tangent Gram matrix at the identity and constructs a whitening map that makes the infinitesimal embedding metric Euclidean. If the Gram is already a scalar multiple of identity, this whitening collapses to a simple global rescale. The finite-difference step is intentionally larger than the ~1e-6 e3nn projection residuals, so the reported Gram reflects the embedding metric rather than projector noise.


In [14]:
def so3_tangent_generators(dtype=dtype, device=device):
    """Infinitesimal active SO(3) generators about x, y, z."""
    Sx = torch.tensor(
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        dtype=dtype,
        device=device,
    )
    Sy = torch.tensor(
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        dtype=dtype,
        device=device,
    )
    Sz = torch.tensor(
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
        dtype=dtype,
        device=device,
    )
    return [Sx, Sy, Sz]


def _as_batched_feature(x: torch.Tensor) -> torch.Tensor:
    return x.unsqueeze(0) if x.dim() == 1 else x


def finite_difference_tangent(embed_fn, eps: float = 1e-3) -> torch.Tensor:
    """Return a (3, C) tangent matrix for Phi(exp(t S_i)) at identity."""
    tangents = []
    for S in so3_tangent_generators():
        Rp = torch.matrix_exp(float(eps) * S).unsqueeze(0)
        Rm = torch.matrix_exp(-float(eps) * S).unsqueeze(0)
        fp = _as_batched_feature(embed_fn(Rp))[0]
        fm = _as_batched_feature(embed_fn(Rm))[0]
        tangents.append((fp - fm) / (2.0 * float(eps)))
    return torch.stack(tangents, dim=0)


def tangent_whitening_map(tangent: torch.Tensor, target_scale: float = 1.0) -> torch.Tensor:
    """
    Build B so row-vector features Phi_iso = Phi @ B have identity tangent Gram.

    The map acts only on the tangent row space and leaves its orthogonal complement
    unchanged. This makes the local metric isotropic without assuming anything
    about how the invariant seed was obtained.
    """
    G = tangent @ tangent.T
    G = 0.5 * (G + G.T)
    evals, Q = torch.linalg.eigh(G)
    evals = evals.clamp_min(1e-14)
    G_inv = Q @ torch.diag(1.0 / evals) @ Q.T
    G_invsqrt = Q @ torch.diag(1.0 / torch.sqrt(evals)) @ Q.T
    tangent_target = float(target_scale) * (G_invsqrt @ tangent)

    C = tangent.shape[1]
    I_C = torch.eye(C, dtype=tangent.dtype, device=tangent.device)
    row_projector = tangent.T @ G_inv @ tangent
    return tangent.T @ G_inv @ tangent_target + (I_C - row_projector)


def local_isometry_prepare(name: str, embed_fn, eps: float = 1e-3, target_scale: float = 1.0):
    """Compute tangent Gram, scalar scale, and a full tangent-whitening map."""
    tangent = finite_difference_tangent(embed_fn, eps=eps)
    gram = tangent @ tangent.T
    gram = 0.5 * (gram + gram.T)
    evals = torch.linalg.eigvalsh(gram)
    mean_eval = evals.mean().clamp_min(1e-14)
    scalar_rescale = 1.0 / torch.sqrt(mean_eval)
    anisotropy = torch.linalg.norm(gram / mean_eval - torch.eye(3, dtype=gram.dtype, device=gram.device))

    B = tangent_whitening_map(tangent, target_scale=target_scale)
    iso_tangent = tangent @ B
    iso_gram = 0.5 * (iso_tangent @ iso_tangent.T + (iso_tangent @ iso_tangent.T).T)

    print(f"=== {name} ===")
    print("feature_dim:", int(tangent.shape[1]))
    print("tangent Gram before:")
    print(gram)
    print("Gram eigenvalues before:", evals.tolist())
    print("scalar-only rescale 1/sqrt(mean eig):", float(scalar_rescale.item()))
    print("relative anisotropy ||G/mean(G)-I||:", float(anisotropy.item()))
    print("tangent Gram after whitening:")
    print(iso_gram)
    print()
    return {
        "tangent": tangent,
        "gram": gram,
        "evals": evals,
        "scalar_rescale": scalar_rescale,
        "anisotropy": anisotropy,
        "whitening": B,
        "iso_gram": iso_gram,
    }


def _rank4_flat(v: torch.Tensor) -> torch.Tensor:
    p2 = (v.unsqueeze(-1) * v.unsqueeze(-2)).reshape(*v.shape[:-1], 9)
    return (p2.unsqueeze(-1) * p2.unsqueeze(-2)).reshape(*v.shape[:-1], 81)


cart_seed_raw_4e_for_metric = y[0, slices[idx_4e]]
cart_seed_norm_4e = cart_seed_raw_4e_for_metric.norm().clamp_min(1e-12)
orbit_t_for_metric = orbit.transpose(0, 1).contiguous()
reyn_seed_4e_metric = reyn_seed_4e.clone()
if torch.dot(cart_seed_4e, reyn_seed_4e_metric) < 0.0:
    reyn_seed_4e_metric = -reyn_seed_4e_metric


def fcc_cartesian_rank4_4e_feature(R: torch.Tensor) -> torch.Tensor:
    """Cartesian tensor route, reduced to the nonzero FCC 4e block."""
    v = torch.matmul(R, orbit_t_for_metric).transpose(-2, -1)
    x = _rank4_flat(v).mean(dim=-2)
    y_cart = x @ proj
    return y_cart[..., slices[idx_4e]] / cart_seed_norm_4e


def fcc_reynolds_4e_feature(R: torch.Tensor) -> torch.Tensor:
    """Direct Reynolds route using the aligned 4e invariant seed."""
    return o3.Irrep("4e").D_from_matrix(R) @ reyn_seed_4e_metric


O_cart_iso = local_isometry_prepare("O Cartesian rank-4 -> 4e", fcc_cartesian_rank4_4e_feature)
O_reyn_iso = local_isometry_prepare("O direct Reynolds 4e", fcc_reynolds_4e_feature)

q_check = random_unit_quats(32, seed=123, dtype=dtype)
R_check = o3.quaternion_to_matrix(q_check)
method_diff = fcc_cartesian_rank4_4e_feature(R_check) - fcc_reynolds_4e_feature(R_check)
print("max |O Cartesian feature - O Reynolds feature|:", method_diff.abs().max().item())
print("max |O Cartesian Gram - O Reynolds Gram|:", (O_cart_iso["gram"] - O_reyn_iso["gram"]).abs().max().item())


=== O Cartesian rank-4 -> 4e ===
feature_dim: 9
tangent Gram before:
tensor([[6.66663e+00, 5.11233e-30, 2.05432e-29],
        [5.11233e-30, 6.66663e+00, 0.00000e+00],
        [2.05432e-29, 0.00000e+00, 6.66663e+00]], dtype=torch.float64)
Gram eigenvalues before: [6.666630874490109, 6.666630874490109, 6.666630969060382]
scalar-only rescale 1/sqrt(mean eig): 0.387299373378026
relative anisotropy ||G/mean(G)-I||: 1.158250769994532e-08
tangent Gram after whitening:
tensor([[ 1.00000e+00, -1.65944e-30, -6.66825e-30],
        [-1.65944e-30,  1.00000e+00,  3.29104e-16],
        [-6.66825e-30,  3.29104e-16,  1.00000e+00]], dtype=torch.float64)

=== O direct Reynolds 4e ===
feature_dim: 9
tangent Gram before:
tensor([[ 6.66663e+00,  1.24859e-03,  7.80223e-08],
        [ 1.24859e-03,  6.66913e+00, -5.68383e-13],
        [ 7.80223e-08, -5.68383e-13,  6.66663e+00]], dtype=torch.float64)
Gram eigenvalues before: [6.666113923670987, 6.666631031818187, 6.6696446823709525]
scalar-only rescale 1/sqrt(m

## 10. Direct comparison of the two methods

| Question | Cartesian tensor route | Direct Reynolds route |
|---|---|---|
| Starting object | A chosen geometric orbit, e.g. `u=e1`, rank `4` | A chosen irrep degree `l` |
| Main operation | Build `T_n(R)=mean_g (Rgu)^⊗n` | Build `P_l=mean_g D^l(g)` |
| Decomposition | Decompose Cartesian tensor into SO(3) irreps | Already inside one SO(3) irrep |
| Scalar handling | `0e` appears but is dropped as constant | `0e` appears only if you ask for `l=0` |
| Why FCC `2e` vanishes | The cubic orbit has isotropic second trace | Reynolds projector has no eigenvalue-1 vector in `2e` |
| Why FCC `4e` remains | Rank-4 cubic anisotropy survives | `P_4` has a 1D invariant eigenspace |
| Can see `6e`? | Not from rank 4 | Yes, if you ask for `l=6` |
| Relation for FCC rank 4 | Produces the same `4e` A1 line | Produces that line directly |

## 11. Mapping back to the production OCRP encoder

The current encoder uses the Cartesian route:

1. FCC chooses `group_name = "O"`, rank `4`, reference direction `e1`.
2. It builds orbit directions `{g e1}`.
3. It builds the raw rank-4 orbit tensor by averaging tensor powers.
4. It projects the tensor into nonscalar irreps using `CartesianTensor`.
5. It optionally prunes irreps with no A1 support.

The direct Reynolds method is conceptually cleaner if your goal is simply:

> find all cubic-invariant seeds inside each `l` irrep.

The Cartesian method is more geometric:

> build a descriptor from physical crystal-axis orbits, then decompose whatever irreps that descriptor contains.

For FCC rank 4, both meet at the same nontrivial result: a single `4e` cubic A1 seed.

## Takeaways for `O` and `Oh`

1. The rank-4 Cartesian FCC tensor generically decomposes as `0e + 2e + 4e`.
2. The scalar `0e` is constant and is dropped.
3. The cubic group average makes the `2e` component zero.
4. The meaningful FCC rank-4 descriptor is therefore `4e`.
5. Direct Reynolds projection finds the same `4e` invariant line directly.
6. Direct Reynolds also reveals higher cubic invariant lines, e.g. `6e`, that the rank-4 tensor method cannot produce.
7. For even-parity descriptors, `O` and `Oh` give the same invariant seeds. Odd-parity invariant seeds vanish under `Oh` because of inversion.

# Part II — HCP invariant irreps: `D6` and `D6h`

Now we repeat the same comparison for the HCP/hexagonal case.

The current OCRP HCP encoder uses the proper rotational subgroup `D6` internally. The dataset symmetry is usually reported as `D6h`, which also includes inversion. For the even-parity Cartesian tensors used here, `D6` and `D6h` give the same invariant seeds.

The HCP local-isometric embedding differs from FCC in an important way:

- FCC uses one rank-4 orbit descriptor.
- HCP uses two orbit descriptors:
  1. a rank-2 axial descriptor from the hexagonal/c-axis direction `e3`;
  2. a rank-6 basal descriptor from the basal direction `e1`.

In the code's `z_axis` convention:

$$
u_2=e_3,\qquad u_6=e_1.
$$

The final HCP irrep layout is

$$
2\times 2e + 1\times 4e + 1\times 6e,
$$

where the two `2e` copies come from different Cartesian tensor blocks.

## 12. Build `D6` and `D6h`

`D6` is the 12-element proper rotational hexagonal group. `D6h` adds inversion, so we model it as

$$
D_{6h}=D_6\cup(-D_6).
$$

For even-parity descriptors, the inversion half acts as `+1`, so the invariant seeds are unchanged. Odd-parity seeds are canceled by the `D6h` average.

In [15]:
G_D6 = dihedral_group_D6_zaxis(dtype=dtype, device=device)  # [12, 3, 3]
G_D6h = torch.cat([G_D6, -G_D6], dim=0)                    # [24, 3, 3]

print("|D6|  =", G_D6.shape[0])
print("|D6h| =", G_D6h.shape[0])
print("det(D6) unique:", torch.unique(torch.round(torch.linalg.det(G_D6))))
print("det(D6h) unique:", torch.unique(torch.round(torch.linalg.det(G_D6h))))

|D6|  = 12
|D6h| = 24
det(D6) unique: tensor([1.00000e+00], dtype=torch.float64)
det(D6h) unique: tensor([-1.00000e+00, 1.00000e+00], dtype=torch.float64)


## 13. Direct Reynolds multiplicities for HCP-relevant bands

The HCP Cartesian descriptors used here can only produce even-rank polar tensor irreps:

$$
0e,\ 2e,\ 4e,\ 6e.
$$

So we inspect those bands directly with the Reynolds projector

$$
P_l=\frac1{|G|}\sum_{g\in G}D^{(l)}(g).
$$

Expected HCP result:

- `2e` has one invariant seed;
- `4e` has one invariant seed;
- `6e` has two independent invariant seeds.

That last point is important: the direct Reynolds method sees a 2D invariant subspace inside `6e`, while the current Cartesian rank-6 basal orbit selects one particular vector inside that 2D space.

In [16]:
hcp_relevant_l = [0, 2, 4, 6]

print("D6, even parity")
for l in hcp_relevant_l:
    _, evals, U = invariant_basis(G_D6, l=l, parity="e")
    print(f"D6,  {l}e: invariant_dim={U.shape[1]}, top_eval={evals[-1].item(): .8f}")

print("\nD6h, even parity")
for l in hcp_relevant_l:
    _, evals, U = invariant_basis(G_D6h, l=l, parity="e")
    print(f"D6h, {l}e: invariant_dim={U.shape[1]}, top_eval={evals[-1].item(): .8f}")

print("\nD6h, odd parity")
for l in hcp_relevant_l:
    _, evals, U = invariant_basis(G_D6h, l=l, parity="o")
    print(f"D6h, {l}o: invariant_dim={U.shape[1]}, top_eval={evals[-1].item(): .8e}")

D6, even parity
D6,  0e: invariant_dim=1, top_eval= 1.00000000
D6,  2e: invariant_dim=1, top_eval= 1.00000000
D6,  4e: invariant_dim=1, top_eval= 1.00000000
D6,  6e: invariant_dim=2, top_eval= 1.00000000

D6h, even parity
D6h, 0e: invariant_dim=1, top_eval= 1.00000000
D6h, 2e: invariant_dim=1, top_eval= 1.00000000
D6h, 4e: invariant_dim=1, top_eval= 1.00000000
D6h, 6e: invariant_dim=2, top_eval= 1.00000000

D6h, odd parity
D6h, 0o: invariant_dim=0, top_eval= 0.00000000e+00
D6h, 2o: invariant_dim=0, top_eval= 1.14853864e-17
D6h, 4o: invariant_dim=0, top_eval= 2.99068334e-17
D6h, 6o: invariant_dim=0, top_eval= 4.64229674e-17


# HCP Method A: Cartesian orbit tensors

The HCP Cartesian route builds two descriptors.

## Rank-2 axial descriptor

The first descriptor uses the c-axis:

$$
u_2=e_3.
$$

The orbit average is

$$
T_2(R)=\frac1{12}\sum_{g\in D_6}(Rg e_3)^{\otimes2}.
$$

At `R=I`, the group sends `e3` to `+e3` or `-e3`. Since the rank is even, signs disappear and the descriptor is essentially

$$
T_2(I)=e_3e_3^T.
$$

Unlike the FCC second trace, this is not isotropic. Its traceless part is nonzero, so HCP naturally has a rank-2 / `2e` signal.

In [17]:
e2_axis = torch.tensor([0.0, 1.0, 0.0], dtype=dtype, device=device)
e3_axis = torch.tensor([0.0, 0.0, 1.0], dtype=dtype, device=device)

u2_hcp = e3_axis
u6_hcp = e1

orbit_u2 = G_D6 @ u2_hcp
unique_u2, counts_u2 = torch.unique(orbit_u2.round(decimals=6), dim=0, return_counts=True)

print("D6 orbit of u2=e3:")
for v, c in zip(unique_u2, counts_u2):
    print(f"{v.tolist()}  count={int(c)}")

T2_hcp = torch.einsum("ga,gb->ab", orbit_u2, orbit_u2) / orbit_u2.shape[0]
T2_iso = torch.trace(T2_hcp) / 3.0 * torch.eye(3, dtype=dtype, device=device)
T2_traceless = T2_hcp - T2_iso

print("\nT2(I):")
print(T2_hcp)
print("\ntraceless part of T2(I):")
print(T2_traceless)
print("\n||traceless T2|| =", T2_traceless.norm().item())

D6 orbit of u2=e3:
[0.0, 0.0, -1.0]  count=6
[0.0, 0.0, 1.0]  count=6

T2(I):
tensor([[0.00000e+00, 0.00000e+00, 0.00000e+00],
        [0.00000e+00, 0.00000e+00, 0.00000e+00],
        [0.00000e+00, 0.00000e+00, 1.00000e+00]], dtype=torch.float64)

traceless part of T2(I):
tensor([[-3.33333e-01, 0.00000e+00, 0.00000e+00],
        [0.00000e+00, -3.33333e-01, 0.00000e+00],
        [0.00000e+00, 0.00000e+00, 6.66667e-01]], dtype=torch.float64)

||traceless T2|| = 0.8164965809277261


## Rank-6 basal descriptor

The second HCP descriptor uses a basal-plane direction:

$$
u_6=e_1.
$$

The orbit average is

$$
T_6(R)=\frac1{12}\sum_{g\in D_6}(Rg e_1)^{\otimes6}.
$$

At `R=I`, the orbit consists of six basal directions separated by 60 degrees, with signs duplicated. Since the rank is six, signs again disappear.

A generic symmetric rank-6 Cartesian tensor decomposes into

$$
0e\oplus2e\oplus4e\oplus6e.
$$

The scalar `0e` is dropped, leaving possible `2e`, `4e`, and `6e` parts.

In [18]:
def tensor_power_flat(v: torch.Tensor, rank: int) -> torch.Tensor:
    """Flatten v^⊗rank for rank 2, 4, or 6."""
    if rank == 1:
        return v
    p2 = (v.unsqueeze(-1) * v.unsqueeze(-2)).reshape(*v.shape[:-1], 9)
    if rank == 2:
        return p2
    p4 = (p2.unsqueeze(-1) * p2.unsqueeze(-2)).reshape(*v.shape[:-1], 81)
    if rank == 4:
        return p4
    if rank == 6:
        return (p4.unsqueeze(-1) * p2.unsqueeze(-2)).reshape(*v.shape[:-1], 729)
    raise ValueError(f"unsupported rank={rank}")


orbit_u6 = G_D6 @ u6_hcp
unique_u6, counts_u6 = torch.unique(orbit_u6.round(decimals=6), dim=0, return_counts=True)

print("D6 orbit of u6=e1:")
for v, c in zip(unique_u6, counts_u6):
    print(f"{v.tolist()}  count={int(c)}")

x6_identity_flat = tensor_power_flat(orbit_u6, 6).mean(dim=0)
proj6_hcp, irreps6_hcp = _build_nonscalar_projector(
    rank=6,
    formula=full_symmetry_formula(6),
    dtype=dtype,
    device=device,
)
y6_identity = x6_identity_flat.reshape(1, -1) @ proj6_hcp

print("\nrank-6 nonscalar irreps:", irreps6_hcp)
for sl, mul_ir in zip(irreps6_hcp.slices(), irreps6_hcp):
    block = y6_identity[0, sl]
    print(f"{mul_ir}: norm={block.norm().item():.8e}, max_abs={block.abs().max().item():.8e}")

D6 orbit of u6=e1:
[-1.0, 0.0, 0.0]  count=2
[-0.5, -0.866025, 0.0]  count=2
[-0.5, 0.866025, 0.0]  count=2
[0.5, -0.866025, 0.0]  count=2
[0.5, 0.866025, 0.0]  count=2
[1.0, 0.0, 0.0]  count=2

rank-6 nonscalar irreps: 1x2e+1x4e+1x6e
1x2e: norm=3.45032778e-01, max_abs=2.98807151e-01
1x4e: norm=2.09358948e-01, max_abs=1.54823028e-01
1x6e: norm=1.94971972e-01, max_abs=1.60151922e-01


Now we build the rank-2 projector as well and collect the identity seeds from both HCP Cartesian descriptors.

In [19]:
proj2_hcp, irreps2_hcp = _build_nonscalar_projector(
    rank=2,
    formula=full_symmetry_formula(2),
    dtype=dtype,
    device=device,
)
x2_identity_flat = tensor_power_flat(orbit_u2, 2).mean(dim=0)
y2_identity = x2_identity_flat.reshape(1, -1) @ proj2_hcp

print("rank-2 nonscalar irreps:", irreps2_hcp)
for sl, mul_ir in zip(irreps2_hcp.slices(), irreps2_hcp):
    block = y2_identity[0, sl]
    print(f"{mul_ir}: norm={block.norm().item():.8e}, max_abs={block.abs().max().item():.8e}")

hcp_cartesian_blocks = [
    {
        "name": "rank2_axial_u=e3",
        "rank": 2,
        "orbit": orbit_u2,
        "proj": proj2_hcp,
        "irreps": irreps2_hcp,
        "y_identity": y2_identity,
    },
    {
        "name": "rank6_basal_u=e1",
        "rank": 6,
        "orbit": orbit_u6,
        "proj": proj6_hcp,
        "irreps": irreps6_hcp,
        "y_identity": y6_identity,
    },
]

rank-2 nonscalar irreps: 1x2e
1x2e: norm=8.16496578e-01, max_abs=7.07106769e-01


# HCP Method B: direct Reynolds projection

The direct Reynolds method gives the full invariant seed subspace for each chosen irrep.

The Cartesian method gives one seed vector per Cartesian descriptor block and per irrep produced by that block.

So the right comparison is not always "one Cartesian seed equals the whole Reynolds subspace." Instead:

> each Cartesian seed should lie inside the Reynolds invariant subspace for its `l`.

For HCP `6e`, the Reynolds invariant subspace is 2D, but the current rank-6 basal orbit contributes only one `6e` copy, i.e. one direction inside that 2D invariant subspace.

In [20]:
def reynolds_subspace_alignment(group_mats, l: int, seed: torch.Tensor):
    """Measure how well a seed lies in the Reynolds invariant subspace."""
    seed = seed / seed.norm().clamp_min(1e-12)
    _, evals, U = invariant_basis(group_mats, l=l, parity="e")
    proj_seed = U @ (U.T @ seed)
    residual = (seed - proj_seed).norm().item()
    coords = U.T @ seed
    return evals, U, residual, coords


for item in hcp_cartesian_blocks:
    print("\n===", item["name"], "===")
    for sl, mul_ir in zip(item["irreps"].slices(), item["irreps"]):
        _, ir = mul_ir
        seed_raw = item["y_identity"][0, sl]
        evals, U, residual, coords = reynolds_subspace_alignment(G_D6, ir.l, seed_raw)
        print(
            f"{mul_ir}: seed_norm={seed_raw.norm().item():.8e}, "
            f"Reynolds_dim={U.shape[1]}, residual_to_subspace={residual:.3e}, "
            f"coords={coords.tolist()}"
        )


=== rank2_axial_u=e3 ===
1x2e: seed_norm=8.16496578e-01, Reynolds_dim=1, residual_to_subspace=7.671e-08, coords=[0.9999999999999968]

=== rank6_basal_u=e1 ===
1x2e: seed_norm=3.45032778e-01, Reynolds_dim=1, residual_to_subspace=5.546e-08, coords=[-0.9999999999999982]
1x4e: seed_norm=2.09358948e-01, Reynolds_dim=1, residual_to_subspace=8.529e-08, coords=[0.9999999999999966]
1x6e: seed_norm=1.94971972e-01, Reynolds_dim=2, residual_to_subspace=1.021e-07, coords=[0.43003333858558124, -0.9028130081611188]


The residuals should be tiny. That means the HCP Cartesian seeds are valid D6-invariant Reynolds seeds.

But note the structural difference:

- rank-2 axial descriptor gives one `2e` seed;
- rank-6 basal descriptor gives another `2e` seed, one `4e` seed, and one `6e` seed;
- direct Reynolds says the entire `6e` invariant subspace is 2D, but this particular Cartesian construction only uses one vector from it.

## 13b. Local-isometry preparation for `D6`

The same local-isometry preparation applies to HCP/`D6`.

Again, the answer does **not** depend on whether the invariant seeds were obtained by tensor-product/Cartesian decomposition or by Reynolds projection. It depends on the final chosen feature map and its block weights.

For `D6`, this distinction matters more than in FCC:

- the current Cartesian route uses the rank-2 axial seed plus the rank-6 basal seeds, giving the production-style feature space `2x2e + 1x4e + 1x6e`;
- the full direct Reynolds fixed subspaces through `l=6` would instead give `1x2e + 1x4e + 2x6e`.

Both are valid crystal-invariant feature maps, but they are not the same map. The local metric must be measured and calibrated for whichever one we actually use.


In [21]:
def _hcp_rank_beta(rank: int) -> float:
    if int(rank) == 2:
        return 1.0 / math.sqrt(24.0)
    if int(rank) == 6:
        return 2.0 * math.sqrt(2.0) / 3.0
    raise ValueError(f"unexpected HCP rank={rank}")


def _hcp_orbit_average_flat_for_R(R: torch.Tensor, orbit_dirs: torch.Tensor, rank: int) -> torch.Tensor:
    v = torch.matmul(R, orbit_dirs.transpose(0, 1).contiguous()).transpose(-2, -1)
    return tensor_power_flat(v, rank).mean(dim=-2)


def hcp_cartesian_seed_specs_from_blocks():
    """Seeds/scales selected by the Cartesian HCP construction."""
    specs = []
    for item in hcp_cartesian_blocks:
        beta = _hcp_rank_beta(item["rank"])
        for sl, mul_ir in zip(item["irreps"].slices(), item["irreps"]):
            _, ir = mul_ir
            seed_raw = item["y_identity"][0, sl]
            seed_norm = seed_raw.norm()
            if float(seed_norm.item()) <= 1e-10:
                continue
            specs.append(
                {
                    "name": f"{item['name']}::{mul_ir}",
                    "l": int(ir.l),
                    "seed": seed_raw / seed_norm,
                    "scale": float(beta) * float(seed_norm.item()),
                    "rank": int(item["rank"]),
                    "slice": sl,
                    "item": item,
                }
            )
    return specs


def feature_from_seed_specs(R: torch.Tensor, specs: list[dict]) -> torch.Tensor:
    outs = []
    for spec in specs:
        block = o3.Irrep(f"{spec['l']}e").D_from_matrix(R) @ spec["seed"]
        outs.append(spec["scale"] * block)
    return torch.cat(outs, dim=-1)


def d6_cartesian_tensor_feature(R: torch.Tensor) -> torch.Tensor:
    """Production-style D6 tensor route with the same rank betas as local_iso_embedding.py."""
    outs = []
    for item in hcp_cartesian_blocks:
        beta = _hcp_rank_beta(item["rank"])
        x = _hcp_orbit_average_flat_for_R(R, item["orbit"], item["rank"])
        y_item = x @ item["proj"]
        for sl, mul_ir in zip(item["irreps"].slices(), item["irreps"]):
            seed_raw = item["y_identity"][0, sl]
            if float(seed_raw.norm().item()) <= 1e-10:
                continue
            outs.append(float(beta) * y_item[..., sl])
    return torch.cat(outs, dim=-1)


def d6_full_reynolds_feature(R: torch.Tensor, ls=(2, 4, 6)) -> torch.Tensor:
    """Full direct Reynolds fixed subspaces through the requested l values."""
    outs = []
    batch_shape = R.shape[:-2]
    for l in ls:
        _, _, U = invariant_basis(G_D6, l=int(l), parity="e")
        if U.shape[1] == 0:
            continue
        block = o3.Irrep(f"{int(l)}e").D_from_matrix(R) @ U  # batch + (2l+1, r_l)
        # Copies-first flattening: r_l copies of the l irrep.
        block = block.transpose(-1, -2).reshape(*batch_shape, -1)
        outs.append(block)
    return torch.cat(outs, dim=-1)


d6_cart_seed_specs = hcp_cartesian_seed_specs_from_blocks()
print("D6 Cartesian-selected seeds/scales:")
for spec in d6_cart_seed_specs:
    print(f"  {spec['name']}: l={spec['l']}, scale={float(spec['scale']):.8e}")
print()


def d6_reynolds_matching_cartesian_feature(R: torch.Tensor) -> torch.Tensor:
    """Direct-Reynolds evaluation using exactly the Cartesian-selected seeds."""
    return feature_from_seed_specs(R, d6_cart_seed_specs)


D6_cart_iso = local_isometry_prepare("D6 Cartesian rank-2/rank-6 selected feature", d6_cartesian_tensor_feature)
D6_reyn_match_iso = local_isometry_prepare("D6 Reynolds using Cartesian-selected seeds", d6_reynolds_matching_cartesian_feature)
D6_reyn_full_iso = local_isometry_prepare("D6 full Reynolds fixed subspaces l=2,4,6", d6_full_reynolds_feature)

q_check = random_unit_quats(32, seed=456, dtype=dtype)
R_check = o3.quaternion_to_matrix(q_check)
match_diff = d6_cartesian_tensor_feature(R_check) - d6_reynolds_matching_cartesian_feature(R_check)
print("max |D6 Cartesian feature - matching Reynolds feature|:", match_diff.abs().max().item())
print("max |D6 Cartesian Gram - matching Reynolds Gram|:", (D6_cart_iso["gram"] - D6_reyn_match_iso["gram"]).abs().max().item())
print("D6 full Reynolds feature_dim:", int(d6_full_reynolds_feature(torch.eye(3, dtype=dtype, device=device).unsqueeze(0)).shape[-1]))


D6 Cartesian-selected seeds/scales:
  rank2_axial_u=e3::1x2e: l=2, scale=1.66666666e-01
  rank6_basal_u=e1::1x2e: l=2, scale=3.25300022e-01
  rank6_basal_u=e1::1x4e: l=4, scale=1.97385509e-01
  rank6_basal_u=e1::1x6e: l=6, scale=1.83821338e-01

=== D6 Cartesian rank-2/rank-6 selected feature ===
feature_dim: 32
tangent Gram before:
tensor([[9.99996e-01, 0.00000e+00, 0.00000e+00],
        [0.00000e+00, 9.99996e-01, 0.00000e+00],
        [0.00000e+00, 0.00000e+00, 9.99988e-01]], dtype=torch.float64)
Gram eigenvalues before: [0.9999879917100756, 0.9999960282515521, 0.9999960533809037]
scalar-only rescale 1/sqrt(mean eig): 1.0000033211261228
relative anisotropy ||G/mean(G)-I||: 6.5721353278501704e-06
tangent Gram after whitening:
tensor([[1.00000e+00, 0.00000e+00, 0.00000e+00],
        [0.00000e+00, 1.00000e+00, 0.00000e+00],
        [0.00000e+00, 0.00000e+00, 1.00000e+00]], dtype=torch.float64)

=== D6 Reynolds using Cartesian-selected seeds ===
feature_dim: 32
tangent Gram before:
tensor

## 14. Real HCP data check: Ti–Al quaternions start passive

Now we repeat the real-data test using the HCP/Ti–Al dataset.

Again, the stored quaternions are treated as passive scalar-first quaternions:

$$
q_\mathrm{passive}=[w,x,y,z].
$$

Before evaluating either irrep construction, we convert them to active convention:

$$
q_\mathrm{active}=\overline{q_\mathrm{passive}}=[w,-x,-y,-z].
$$

In [22]:
def find_real_hcp_passive_file():
    """Find a real HCP/Ti-Al quaternion block from the local dataset mount."""
    candidate_roots = [
        Path("/data/home/umang/Materials/Materials_data_mount/datasets/Ti_Al_1pct_QSR_x4"),
        Path("/data/home/umang/Materials/Materials_data_mount/datasets/h200_datasets/Ti_Al_1pct_QSR_x4"),
    ]
    for root in candidate_roots:
        info_path = root / "dataset_info.json"
        if not info_path.exists():
            continue
        for split in ("Test", "Val", "Train"):
            for which in ("HR_Data", "LR_Data"):
                files = sorted((root / split / which).glob("*.npy"))
                if files:
                    return root, info_path, files[0]
    raise FileNotFoundError(
        "Could not find Ti_Al_1pct_QSR_x4 under the expected dataset roots. "
        "Update candidate_roots in this cell to point at your HCP dataset."
    )


hcp_root, hcp_info_path, hcp_npy_path = find_real_hcp_passive_file()
hcp_info = json.loads(hcp_info_path.read_text())
hcp_arr = np.load(hcp_npy_path)
hcp_q_hwc = to_hwc4(hcp_arr).astype(np.float64, copy=False)

print("dataset root:", hcp_root)
print("dataset_info:", hcp_info_path)
print("sample file:", hcp_npy_path)
print("symmetry:", hcp_info.get("symmetry"))
print("formatting:", hcp_info.get("formatting"))
print("raw array shape:", hcp_arr.shape, "-> HWC shape:", hcp_q_hwc.shape)

q_hcp_passive_all = torch.from_numpy(hcp_q_hwc.reshape(-1, 4)).to(dtype=dtype, device=device)
q_hcp_passive_all = q_hcp_passive_all / q_hcp_passive_all.norm(dim=-1, keepdim=True).clamp_min(1e-12)
q_hcp_passive_all = torch.where(q_hcp_passive_all[..., :1] < 0.0, -q_hcp_passive_all, q_hcp_passive_all)

gen = torch.Generator(device="cpu")
gen.manual_seed(456)
n_hcp = min(512, q_hcp_passive_all.shape[0])
hcp_idx = torch.randperm(q_hcp_passive_all.shape[0], generator=gen)[:n_hcp]
q_hcp_passive = q_hcp_passive_all[hcp_idx]

q_hcp_active = _quat_conjugate(q_hcp_passive)
R_hcp = _quat_to_matrix_active(q_hcp_active)

print("sampled real passive quaternions:", tuple(q_hcp_passive.shape))
print("active rotation matrices:", tuple(R_hcp.shape))
print("first passive q:", q_hcp_passive[0])
print("first active  q:", q_hcp_active[0])

dataset root: /data/home/umang/Materials/Materials_data_mount/datasets/Ti_Al_1pct_QSR_x4
dataset_info: /data/home/umang/Materials/Materials_data_mount/datasets/Ti_Al_1pct_QSR_x4/dataset_info.json
sample file: /data/home/umang/Materials/Materials_data_mount/datasets/Ti_Al_1pct_QSR_x4/Test/HR_Data/Ti_Al_1pct_QSR_x4_test_hr_x_block_102.npy
symmetry: D6h
formatting: {'Note:': 'The original quaternions stored in the Original_Data files were converted to same scalar-first or last convention, depending on the convert_to_scalar_first flag.', 'convert_to_scalar_first': True, 'normalize': True, 'hemisphere': True, 'reduce_fz': True, 'to_quat_first': False}
raw array shape: (128, 128, 4) -> HWC shape: (128, 128, 4)
sampled real passive quaternions: (512, 4)
active rotation matrices: (512, 3, 3)
first passive q: tensor([7.41472e-01, -1.24503e-01, 6.46990e-01, -1.26975e-01], dtype=torch.float64)
first active  q: tensor([7.41472e-01, 1.24503e-01, -6.46990e-01, 1.26975e-01], dtype=torch.float64)


The next cell evaluates every HCP Cartesian block on real passive-data orientations and compares each resulting irrep block against its direct-Reynolds seed evolution:

$$
f_l(R)=D^{(l)}(R)u_l.
$$

For each Cartesian seed, `u_l` is taken from that block's identity-orientation projection.

In [23]:
def orbit_average_flat_for_R(R: torch.Tensor, orbit_dirs: torch.Tensor, rank: int) -> torch.Tensor:
    """Compute mean_g (R orbit_dirs[g])^⊗rank as a flattened tensor."""
    v = torch.matmul(R, orbit_dirs.transpose(0, 1).contiguous()).transpose(-2, -1)
    return tensor_power_flat(v, rank).mean(dim=-2)


for item in hcp_cartesian_blocks:
    print("\n=== real HCP data:", item["name"], "===")
    x_real = orbit_average_flat_for_R(R_hcp, item["orbit"], item["rank"])
    y_real = x_real @ item["proj"]

    for sl, mul_ir in zip(item["irreps"].slices(), item["irreps"]):
        _, ir = mul_ir
        seed_raw = item["y_identity"][0, sl]
        seed_norm = seed_raw.norm().clamp_min(1e-12)
        seed = seed_raw / seed_norm

        cart_feat = y_real[:, sl] / seed_norm
        reyn_feat = o3.Irrep(f"{ir.l}e").D_from_matrix(R_hcp) @ seed
        diff = cart_feat - reyn_feat

        print(
            f"{mul_ir}: max_abs_error={diff.abs().max().item():.3e}, "
            f"RMS_error={diff.square().mean().sqrt().item():.3e}"
        )


=== real HCP data: rank2_axial_u=e3 ===
1x2e: max_abs_error=7.218e-07, RMS_error=2.762e-07

=== real HCP data: rank6_basal_u=e1 ===
1x2e: max_abs_error=7.211e-07, RMS_error=2.765e-07
1x4e: max_abs_error=1.205e-06, RMS_error=3.825e-07
1x6e: max_abs_error=1.057e-06, RMS_error=3.347e-07


## 14b. Numerical error budget for the HCP comparison

For each HCP Cartesian seed, the exact identity being checked is the same pattern:

$$
\Pi_l\left[\frac{1}{|D_6|}\sum_{g\in D_6}(Rgu)^{\otimes n}\right]
=
D^{(l)}(R)u_l,
$$

where:

- $n=2$, $u=e_3$ for the axial descriptor;
- $n=6$, $u=e_1$ for the basal descriptor;
- $u_l$ is the corresponding identity-orientation Cartesian seed after projection into the `l` irrep block.

The important HCP-specific subtlety is the `6e` block. Direct Reynolds finds a **two-dimensional** invariant subspace in `6e` for `D6`. The rank-6 basal Cartesian tensor does not use every vector in that subspace; it selects one particular normalized seed inside it. Therefore the correct comparison is not “does `6e` have one Reynolds vector?” but rather “does the Cartesian `6e` seed lie inside the Reynolds invariant subspace, and does $D^{(6)}(R)$ evolve that same selected seed?”

The diagnostics below print both pieces:

1. the residual of each Cartesian seed after projection by the Reynolds projector;
2. the real-data feature error between the Cartesian tensor route and the direct $D^{(l)}(R)u_l$ route.


In [24]:
# Numerical diagnostics explaining the HCP residuals above.

I3 = torch.eye(3, dtype=dtype, device=device)
R_hcp_orth_err = (R_hcp @ R_hcp.transpose(-1, -2) - I3).abs().max().item()
R_hcp_det_err = (torch.linalg.det(R_hcp) - 1.0).abs().max().item()

print("HCP numerical error budget")
print(f"  dtype machine epsilon                         : {torch.finfo(dtype).eps:.3e}")
print(f"  max ||R R^T - I||_∞ after passive->active      : {R_hcp_orth_err:.3e}")
print(f"  max |det(R)-1| after passive->active           : {R_hcp_det_err:.3e}")

for item in hcp_cartesian_blocks:
    print(f"\n=== {item['name']} ===")
    x_real = orbit_average_flat_for_R(R_hcp, item["orbit"], item["rank"])
    y_real = x_real @ item["proj"]

    for sl, mul_ir in zip(item["irreps"].slices(), item["irreps"]):
        _, ir = mul_ir
        irrep = o3.Irrep(f"{ir.l}e")
        seed_raw = item["y_identity"][0, sl]
        seed_norm = seed_raw.norm().clamp_min(1e-12)
        seed = seed_raw / seed_norm

        P_l, evals, U = invariant_basis(G_D6, l=ir.l, parity="e")
        seed_projector_residual = (P_l @ seed - seed).norm().item()
        seed_subspace_residual = (seed - U @ (U.T @ seed)).norm().item()

        D_l = irrep.D_from_matrix(R_hcp)
        I_l = torch.eye(2 * ir.l + 1, dtype=dtype, device=device)
        D_l_orth_err = (D_l.transpose(-1, -2) @ D_l - I_l).abs().max().item()

        cart_feat = y_real[:, sl] / seed_norm
        reyn_feat = D_l @ seed
        diff = cart_feat - reyn_feat

        print(f"{mul_ir}")
        print(f"  Reynolds invariant dimension                  : {U.shape[1]}")
        print(f"  top Reynolds eigenvalues                      : {[float(v) for v in evals[-min(3, len(evals)):]]}")
        print(f"  ||P_l seed - seed||_2                         : {seed_projector_residual:.3e}")
        print(f"  residual to Reynolds invariant subspace       : {seed_subspace_residual:.3e}")
        print(f"  max ||D^{ir.l}(R)^T D^{ir.l}(R)-I||_∞          : {D_l_orth_err:.3e}")
        print(f"  observed feature max_abs error                : {diff.abs().max().item():.3e}")
        print(f"  observed feature RMS error                    : {diff.square().mean().sqrt().item():.3e}")
        if U.shape[1] > 1:
            coords = U.T @ seed
            print(f"  coordinates of selected Cartesian seed in invariant subspace: {coords.tolist()}")

# Same convention guard as FCC: agreement of the two methods is not a convention proof.
R_hcp_if_passive_misread_as_active = _quat_to_matrix_active(q_hcp_passive)
print("\nConvention guard")
for item in hcp_cartesian_blocks:
    x_wrong = orbit_average_flat_for_R(R_hcp_if_passive_misread_as_active, item["orbit"], item["rank"])
    y_wrong = x_wrong @ item["proj"]
    x_correct = orbit_average_flat_for_R(R_hcp, item["orbit"], item["rank"])
    y_correct = x_correct @ item["proj"]
    print(f"  {item['name']}")
    for sl, mul_ir in zip(item["irreps"].slices(), item["irreps"]):
        _, ir = mul_ir
        seed_raw = item["y_identity"][0, sl]
        seed_norm = seed_raw.norm().clamp_min(1e-12)
        seed = seed_raw / seed_norm
        cart_wrong = y_wrong[:, sl] / seed_norm
        reyn_wrong = o3.Irrep(f"{ir.l}e").D_from_matrix(R_hcp_if_passive_misread_as_active) @ seed
        cart_correct = y_correct[:, sl] / seed_norm
        print(
            f"    {mul_ir}: wrong internal max_abs={((cart_wrong - reyn_wrong).abs().max().item()):.3e}, "
            f"wrong-vs-correct feature shift={((cart_wrong - cart_correct).abs().max().item()):.3e}"
        )


HCP numerical error budget
  dtype machine epsilon                         : 2.220e-16
  max ||R R^T - I||_∞ after passive->active      : 6.661e-16
  max |det(R)-1| after passive->active           : 8.882e-16

=== rank2_axial_u=e3 ===
1x2e
  Reynolds invariant dimension                  : 1


  top Reynolds eigenvalues                      : [1.3188011014917048e-08, 6.339722621035277e-08, 0.999999999999903]
  ||P_l seed - seed||_2                         : 7.671e-08
  residual to Reynolds invariant subspace       : 7.671e-08
  max ||D^2(R)^T D^2(R)-I||_∞          : 1.754e-14
  observed feature max_abs error                : 7.218e-07
  observed feature RMS error                    : 2.762e-07

=== rank6_basal_u=e1 ===
1x2e
  Reynolds invariant dimension                  : 1
  top Reynolds eigenvalues                      : [1.3188011014917048e-08, 6.339722621035277e-08, 0.999999999999903]
  ||P_l seed - seed||_2                         : 5.546e-08
  residual to Reynolds invariant subspace       : 5.546e-08
  max ||D^2(R)^T D^2(R)-I||_∞          : 1.754e-14
  observed feature max_abs error                : 7.211e-07
  observed feature RMS error                    : 2.765e-07
1x4e
  Reynolds invariant dimension                  : 1
  top Reynolds eigenvalues                  

## HCP takeaways for `D6` and `D6h`

1. The HCP encoder uses **two Cartesian orbit tensors**, not one:
   - rank-2 axial/c-axis descriptor;
   - rank-6 basal descriptor.

2. The rank-2 axial descriptor produces a genuine `2e` signal because the second-order tensor `e3 e3^T` has a nonzero traceless part.

3. The rank-6 basal descriptor produces `2e + 4e + 6e` after dropping the scalar.

4. Direct Reynolds projection says the relevant D6 invariant seed dimensions are:
   - `2e`: one invariant seed;
   - `4e`: one invariant seed;
   - `6e`: two invariant seeds.

5. The Cartesian rank-6 basal descriptor selects **one particular `6e` seed** inside the 2D Reynolds-invariant `6e` subspace. It does not span both `6e` invariant directions by itself.

6. For even-parity tensor descriptors, `D6` and `D6h` give the same results. Odd-parity seeds vanish under `D6h` because inversion cancels them.

7. The real Ti–Al data check uses passive scalar-first quaternions, converts them to active by conjugation, and verifies that the Cartesian and Reynolds constructions agree on actual HCP orientations.